# DeBCR API tutorial
## Train DeBCR model on pre-processed data

This notebook shows how to train DeBCR model to restore low-quality microscopy data.

To achieve that you need **pre-processed training/validation data**, consisting of low-quality input and high-quality ground truth, both normalized and patched.

Please find on the DeBCR GitHub page links to:
- notebook tutorial on raw data pre-processing protocol; 
- samples, i.e. examples of pre-processed training/validation data.

In [ ]:
import debcr

### Load training/validation data

Set file path to your actual pre-processed input data (in NPZ or NPY format).

For sample data:
- training data: ```/path/to/examples/DATASET/data/DATASET_train.npz```
- validation data: ```/path/to/examples/DATASET/data/DATASET_val.npz```

In [ ]:
train_data_filepath = '/path/to/data/train.npz'
val_data_filepath = '/path/to/data/val.npz'
train_data_filepath, val_data_filepath

Load training/validation data

In [ ]:
data_train = debcr.data.load(train_data_filepath)
data_val = debcr.data.load(val_data_filepath)

The example training/validaton data is provided as multi-array NPZ, which contains two arrays:
- "low" - input data (low-quality data to be improved)
- "gt" - ground-truth data for comparison

You can check the filenames as below:

In [ ]:
data_train.files, data_val.files

### Visualize loaded training/validation data

for sample training data

In [ ]:
debcr.data.show(
    data = [data_train["low"], data_train["gt"]],
    slices = [-1, -1, -1], # -1 is to pick a random slice
    titles = ['train: input', 'train: ground truth'],
    transpose = True
)

for sample validation data

In [ ]:
debcr.data.show(
    data = [data_val["low"], data_val["gt"]],
    slices = [-1, -1, -1], # -1 is to pick a random slice
    titles = ['validation: input', 'validation: ground truth'],
    transpose = True
)

### Setup training configuration

Next, we need to set the configuration parameters for the training. You can load defaults as below

In [ ]:
config = debcr.config.load()
config

You can adjust configuration, for example

In [ ]:
config['batch_size'] = '16'
config

You can also save configuration as YAML file

In [ ]:
config_path = './config.yaml'
debcr.config.save(config, config_path)

And load it back when needed

In [ ]:
config = debcr.config.load(config_path)
config

### Train model on training data

#### A. Train new model

To train the DeBCR model (from scratch) use the command below. The intermediate checkpoints will be printed and the training will stop automatically.

In [ ]:
debcr_model = debcr.model.train(data_train, data_val, config)

#### B. Continue training model

To continue training from already somewhat trained model, you should first load that model (as described in the prediction tutorial):

In [ ]:
start_model = debcr.model.init(weights_path='/path/to/weights', input_size=128)
start_model.summary()

and the pass that loaded model along with the data and configurations to the training interface

In [ ]:
debcr_model = debcr.model.train(data_train, data_val, config, start_model)

### Test trained model on validation data

Finally, we can try to run the freshly trained model on the validation data

In [ ]:
data_pred = debcr.model.predict(debcr_model, data_val["low"])
data_pred.shape

In [ ]:
debcr.data.show(
    data = [data_val['low'], data_pred, data_val["gt"]],
    slices = [-1, -1],
    titles = ['validation input', 'validation prediction', 'validation ground truth']
)

Further you can use the trained here model to run predictions (see the tutorial link on the GitHub).